# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.### Dataset SourceThe dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed!pip install mlcroissant

## 1. Data LoadingLoad metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlcimport pandas as pd# Define the dataset URLurl = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'# Load the dataset metadatadataset = mlc.Dataset(url)# Access the metadata object for informationprint(f"Dataset Name: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")

## 2. Data OverviewReview available record sets, fields, and their IDs.We will enumerate the available record sets, their `@id`s, and their fields.

In [ ]:
# Discover record sets and their fields using their @id referencesrecord_sets = list(dataset.metadata.record_sets)print(f"Number of record sets: {len(record_sets)}")for idx, record_set in enumerate(record_sets):    print(f"Record Set {idx+1}:")    print(f"  Name: {getattr(record_set, 'name', None)}")    print(f"  @id: {record_set.id}")    fields = list(getattr(record_set, 'fields', []))    print(f"  Number of fields: {len(fields)}")    for field in fields:        print(f"    Field: {getattr(field, 'name', None)} | @id: {field.id}")if len(record_sets) == 0:    print('No record sets are declared at the top level. Attempting to infer from metadata columns and files...')    # Try to explore files and their columns as record sets.    datafiles = getattr(dataset.metadata, 'datafiles', getattr(dataset.metadata, 'files', []))    if hasattr(dataset.metadata, 'datafiles'):        print(f"Number of datafiles: {len(datafiles)}")        for idx, datafile in enumerate(datafiles):            print(f"DataFile {idx+1}:")            print(f"  Name: {getattr(datafile, 'name', None)}")            print(f"  @id: {getattr(datafile, 'id', None)}")            columns = getattr(datafile, 'columns', [])            print(f"  Number of columns: {len(columns)}")            for c in columns:                print(f"    Column: {getattr(c, 'name', None)} | @id: {c.id}")    else:        print('Unable to locate any datafiles/record sets.')

## 3. Data ExtractionLoad data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.We will extract data for each available record set. If none were found in the overview, appropriate error or placeholder output will appear.

In [ ]:
# Attempt to extract records for each record set by @iddataframes = {}if len(record_sets) > 0:    for record_set in record_sets:        rec_id = record_set.id        print(f"Loading records for record set with @id: {rec_id}")        records = list(dataset.records(record_set=rec_id))        dataframes[rec_id] = pd.DataFrame(records)        print(f"  Columns: {dataframes[rec_id].columns.tolist()}")        print(dataframes[rec_id].head())else:    # Try to use a guessed main record set id if possible. The Croissant schema may still expose records via a default or primary record set.    print("No record sets found. Attempting to load records using the default method...")    try:        records = list(dataset.records())        df = pd.DataFrame(records)        dataframes['main'] = df        print(f"Loaded a default/main records set with columns: {df.columns.tolist()}")        print(df.head())    except Exception as e:        print(f"Could not load any records: {e}")

## 4. Exploratory Data Analysis (EDA)Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.This section demonstrates numeric filtering, normalization, and grouping. All references are by `@id` field/column names.

In [ ]:
# *** Choose your main record set and numeric field to analyze. Update these after inspecting above output. ***if len(dataframes) > 0:    # Use the first available dataframe    main_record_set_id = list(dataframes.keys())[0]    df = dataframes[main_record_set_id]    print(f"Working with DataFrame for record set '@id': {main_record_set_id}, columns: {df.columns.tolist()}")    # --- Select a numeric field/column by @id or name ---    numeric_candidates = [col for col in df.columns if df[col].dtype.kind in 'ifc' or pd.api.types.is_numeric_dtype(df[col])]    if len(numeric_candidates) == 0:        print("No numeric fields found for EDA.")    else:        numeric_field_id = numeric_candidates[0]  # Use the first numeric column        print(f"Numeric field selected (by @id or name): {numeric_field_id}")        threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10        filtered_df = df[df[numeric_field_id] > threshold]        print(f"Filtered records with {numeric_field_id} > {threshold}:")        print(filtered_df.head())        # Normalize the numeric field        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()        print(f"Normalized {numeric_field_id} for filtered records:")        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())        # Try grouping by a categorical field        group_candidates = [col for col in df.columns if col != numeric_field_id and df[col].dtype == 'O']        if group_candidates:            group_field = group_candidates[0]            print(f"Grouping by: {group_field}")            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()            print(grouped_df.head())        else:            print("No suitable categorical fields for grouping.")else:    print("No dataframes available for EDA.")

## 5. VisualizationVisualize data distributions or relationships between fields in the dataset.Here we create a simple histogram or barplot based on the numeric field discovered above.

In [ ]:
import matplotlib.pyplot as pltimport seaborn as snsif len(dataframes) > 0 and 'numeric_field_id' in locals():    df = dataframes[main_record_set_id]    plt.figure(figsize=(8,4))    sns.histplot(df[numeric_field_id].dropna(), kde=True)    plt.title(f'Distribution of {numeric_field_id} (@id)')    plt.xlabel(numeric_field_id)    plt.ylabel('Count')    plt.show()    if 'group_field' in locals():        plt.figure(figsize=(10,6))        sns.boxplot(data=df, x=group_field, y=numeric_field_id)        plt.title(f'{numeric_field_id} by {group_field}')        plt.xticks(rotation=45)        plt.show()else:    print('No data available for visualization.')

## 6. ConclusionSummarize key findings and observations from the dataset exploration.* In this notebook, we loaded and explored the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using only programmatic and schema-based field references via Croissant `@id`s.* We inspected record set and field composition, loaded records into Pandas DataFrames, and demonstrated basic filtering, normalization, grouping, and visualization.* For further analysis, review the fields (columns) in each DataFrame, customize groupings, and extend EDA to cover more domain-specific biological questions.